In [1]:
import os
import sys
import xmlrpc.client
import pandas as pd
import logging
from dotenv import load_dotenv
import etl_odoo_incremental as etl
from db_loader import DBLoader
loader = DBLoader()    
tabla = "odoo_apuntes"


desde = etl.ultima_fecha(loader, tabla=tabla)
db, uid, pw, models = etl.conectar_odoo()
# ── Mapeo de campos de account.move.line ──────────────────────────────────────
campos = models.execute_kw(db, uid, pw, 'hr.employee.public', 'fields_get', [],
                           {'attributes': ['string', 'type', 'relation', 'store', 'required']})


2026-06-22 15:40:11,924 - INFO - DB Railway: conexión OK
2026-06-22 15:40:12,470 - INFO - Conexion establecida con Railway PostgreSQL
d:\Desktop\analisis_datos\db_loader.py:357: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=params)
2026-06-22 15:40:12,751 - INFO - Consulta ejecutada — shape: (1, 1)
2026-06-22 15:40:12,752 - INFO - Conexion cerrada
2026-06-22 15:40:13,411 - INFO - Odoo conectado (uid=75)


In [ ]:

df_campos = pd.DataFrame([
    {
        'campo': nombre,
        'etiqueta': info.get('string', ''),
        'tipo': info.get('type', ''),
        'relacion': info.get('relation', ''),
        'almacenado': info.get('store', False),
        'requerido': info.get('required', False),
    }
    for nombre, info in campos.items()
]).sort_values('campo').reset_index(drop=True)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
print(f"Total de campos: {len(df_campos)}")
df_campos

In [4]:
domain = []

fields = [
    "id",
    "name",
    "job_title",
    "parent_id",
    "department_id",
    "work_email",
    "work_phone",
    "company_id",
    "write_date",
]

datos = etl.descargar_modelo_paginado(
    models,
    db,
    uid,
    pw,
    "hr.employee",
    domain,
    fields
)

df = pd.DataFrame(datos)

2026-06-19 16:28:18,110 - INFO -   [hr.employee] 124 registros acumulados


In [5]:
df

,id,name,job_title,parent_id,department_id,work_email,work_phone,company_id,write_date
0,3,MIGUEL ANGEL CUADROS RODRIGUEZ,COORDINADOR DE COMPRAS,"[45, JUAN DAVID ORTIZ ZAPATA]","[53, COMPRAS]",mcuadros@lapocion.com,3162766666,"[8, PCN POCION S.A.S.]",2026-04-22 16:28:33
1,4,ANDRES FELIPE VASQUEZ BONILLA,GERENTE COMERCIAL,"[4, ANDRES FELIPE VASQUEZ BONILLA]","[17, COMERCIAL]",andresvasquez@lapocion.com,3162766666,"[8, PCN POCION S.A.S.]",2026-06-01 15:03:04
2,5,JAIME ADOLFO CORREA SCARPETTA,GERENTE FINANCIERO Y ADMINISTRATIVO,False,"[22, GERENCIA]",jcorrea@lapocion.com,3162766666,"[8, PCN POCION S.A.S.]",2026-04-22 16:28:33
3,6,ELIZABETH RUBIANO,ASISTENTE COMERCIAL,"[45, JUAN DAVID ORTIZ ZAPATA]","[53, COMPRAS]",hoddet2783@hotmail.com,3162766666,"[8, PCN POCION S.A.S.]",2026-05-29 14:13:35
4,8,AIXA CELESTE CUELLAR ILLERA,ASISTENTE DE DESARROLLO E INNOVACION,"[9, DIANA CAROLINA KLINGER]","[18, DESARROLLO]",acuellar@lapocion.com,3162766666,"[8, PCN POCION S.A.S.]",2026-04-22 16:28:33
...,...,...,...,...,...,...,...,...,...
119,215,KAREN JOHANNA PUPIALES PARRA,False,False,False,karenpupiales7@gmail.com,3181963372,"[8, PCN POCION S.A.S.]",2026-06-12 15:38:11
120,216,VALENTINA DUERTO,False,False,False,valentinaduerto1@gmail.com,3181963372,"[8, PCN POCION S.A.S.]",2026-06-12 15:39:28
121,217,LUISA FERNANDA PASTAS VELASCO,False,False,False,luisavelasco07@hotmail.com,3181963372,"[8, PCN POCION S.A.S.]",2026-06-12 15:37:41
122,218,KELLY JOHANA RENGIFO HERRERA,Aprendiz SENA,"[11, ANGELICA MARIA RAMIREZ RODRIGUEZ]","[18, DESARROLLO]",KRENGIFO@LAPOCION.COM,3166828173,"[8, PCN POCION S.A.S.]",2026-06-09 15:47:26


In [9]:
# ── Mapeo de campos de account.move.line ──────────────────────────────────────
campos = models.execute_kw(db, uid, pw, 'hr.employee', 'fields_get', [],
                           {'attributes': ['string', 'type', 'relation', 'store', 'required']})

df_campos = pd.DataFrame([
    {
        'campo': nombre,
        'etiqueta': info.get('string', ''),
        'tipo': info.get('type', ''),
        'relacion': info.get('relation', ''),
        'almacenado': info.get('store', False),
        'requerido': info.get('required', False),
    }
    for nombre, info in campos.items()
]).sort_values('campo').reset_index(drop=True)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
print(f"Total de campos: {len(df_campos)}")


Total de campos: 142


In [11]:
df_campos.to_excel(r'D:\Desktop\empleados_campos.xlsx', index=False)

In [ ]:
# fields = df_campos['campo'].to_list()

# muestra = models.execute_kw(db, uid, pw, 'account.analytic.plan', 'search_read',
#                             [[]], {'fields': fields, 'limit': 5})
# pd.DataFrame(muestra).rename(columns={0: 'valor'})

In [ ]:

# domain = [
#     ["write_date", ">", desde],
#     ["move_id.move_type", "in", ["out_invoice", "out_refund"]],
# ]
# fields = df_campos['campo'].to_list()

# datos = etl.descargar_modelo_paginado(models, db, uid, pw, "account.analytic.plan", fields)

# df = pd.DataFrame(datos)

In [ ]:
# df_campos.to_excel(r'D:\Desktop\campos_account_move_line.xlsx', index=False)

In [ ]:


# domain = [
#     ["write_date", ">", desde],
#     ["move_id.move_type", "in", ["out_invoice", "out_refund"]],
# ]
# fields = [ "x_studio_related_field_9er_1ipkj4lvp", "ref", "matching_number", "analytic_line_ids",
#     "id", "date", "invoice_date", "move_id","move_name", "account_id", "company_id",
#     "partner_id", "quantity", "price_unit", "price_subtotal",
#     "debit", "credit", "balance", "name", "write_date",
# ]

# datos = etl.descargar_modelo_paginado(models, db, uid, pw, "account.move.line", domain, fields)

# df = pd.DataFrame(datos)
# df

2026-06-22 13:13:52,361 - INFO -   [account.move.line] 37 registros acumulados


,id,x_studio_related_field_9er_1ipkj4lvp,ref,matching_number,analytic_line_ids,date,invoice_date,move_id,move_name,account_id,company_id,partner_id,quantity,price_unit,price_subtotal,debit,credit,balance,name,write_date
0,14680422,1053001283,#149495,False,[1907595],2026-06-22,2026-06-22,"[5812967, FE40170 (#149495)]",FE40170,"[7237, 41353801 VENTA DE COSMETICOS GRAVADO 19%]","[8, PCN POCION S.A.S.]","[385392, Derian Prieto]",1.0,157600.0,132436.97,0.0,132436.97,-132436.97,[PCNKIT15] KIT CONTROL GRASA Y REPARACIÓN,2026-06-22 18:10:32
1,14680423,1053001283,#149495,False,[],2026-06-22,2026-06-22,"[5812967, FE40170 (#149495)]",FE40170,"[7202, 24080101 IVA GENERADO EN VENTAS 19%]","[8, PCN POCION S.A.S.]","[385392, Derian Prieto]",0.0,0.0,0.00,0.0,25163.03,-25163.03,19% IVA VENTAS,2026-06-22 18:10:32
2,14680424,1053001283,#149495,4379652,[],2026-06-22,2026-06-22,"[5812967, FE40170 (#149495)]",FE40170,"[6103, 13050501 CLIENTES NACIONALES]","[8, PCN POCION S.A.S.]","[385392, Derian Prieto]",0.0,0.0,0.00,157600.0,0.00,157600.00,41597180543254,2026-06-22 18:10:32
3,14680448,1000835322,#149496,False,"[1907612, 1907611]",2026-06-22,2026-06-22,"[5812976, FE40171 (#149496)]",FE40171,"[7237, 41353801 VENTA DE COSMETICOS GRAVADO 19%]","[8, PCN POCION S.A.S.]","[373417, Valentina Calvo]",1.0,43900.0,36890.76,0.0,36890.76,-36890.76,[TNGL05] EMULGEL RIZOS TONGOLÉ,2026-06-22 18:10:32
4,14680449,1000835322,#149496,False,"[1907614, 1907613]",2026-06-22,2026-06-22,"[5812976, FE40171 (#149496)]",FE40171,"[7237, 41353801 VENTA DE COSMETICOS GRAVADO 19%]","[8, PCN POCION S.A.S.]","[373417, Valentina Calvo]",1.0,58900.0,49495.80,0.0,49495.80,-49495.80,[PCN19] DUTONIC (TONICO CAPILAR),2026-06-22 18:10:32
5,14680450,1000835322,#149496,False,"[1907616, 1907615]",2026-06-22,2026-06-22,"[5812976, FE40171 (#149496)]",FE40171,"[7237, 41353801 VENTA DE COSMETICOS GRAVADO 19%]","[8, PCN POCION S.A.S.]","[373417, Valentina Calvo]",1.0,48900.0,41092.44,0.0,41092.44,-41092.44,[PCN09] TERMOPROTECTOR BRILLO INFINITO,2026-06-22 18:10:32
6,14680451,1000835322,#149496,False,"[1907618, 1907617]",2026-06-22,2026-06-22,"[5812976, FE40171 (#149496)]",FE40171,"[7237, 41353801 VENTA DE COSMETICOS GRAVADO 19%]","[8, PCN POCION S.A.S.]","[373417, Valentina Calvo]",1.0,48900.0,41092.44,0.0,41092.44,-41092.44,[PCN26] SHAMPOO CRECIMIENTO Y CAIDA,2026-06-22 18:10:32
7,14680452,1000835322,#149496,False,[1907619],2026-06-22,2026-06-22,"[5812976, FE40171 (#149496)]",FE40171,"[7237, 41353801 VENTA DE COSMETICOS GRAVADO 19%]","[8, PCN POCION S.A.S.]","[373417, Valentina Calvo]",1.0,44900.0,37731.09,0.0,37731.09,-37731.09,[PCN32] SHAMPOO CONTROL CASPA,2026-06-22 18:10:32
8,14680453,1000835322,#149496,False,"[1907621, 1907620]",2026-06-22,2026-06-22,"[5812976, FE40171 (#149496)]",FE40171,"[7237, 41353801 VENTA DE COSMETICOS GRAVADO 19%]","[8, PCN POCION S.A.S.]","[373417, Valentina Calvo]",1.0,34900.0,29327.73,0.0,29327.73,-29327.73,[PCN11] DOYPACK SHAMPOO,2026-06-22 18:10:32
9,14680454,1000835322,#149496,False,[],2026-06-22,2026-06-22,"[5812976, FE40171 (#149496)]",FE40171,"[7202, 24080101 IVA GENERADO EN VENTAS 19%]","[8, PCN POCION S.A.S.]","[373417, Valentina Calvo]",0.0,0.0,0.00,0.0,44769.74,-44769.74,19% IVA VENTAS,2026-06-22 18:10:32


In [4]:
df.to_excel(r'd:\Desktop\odoo_apuntes.xlsx', index=False)

In [ ]:

# if not datos:
#     logging.info(f"[{tabla}] Sin cambios.")
#     return

# df = pd.DataFrame(datos)
# for col in ["account_id", "partner_id", "move_id"]:
#     df = expandir(df, col)

# loader.preparar_y_cargar(df, tabla)


In [3]:
# ── Mapeo de campos de account.move.line ──────────────────────────────────────
campos = models.execute_kw(db, uid, pw, 'hr.employee', 'fields_get', [],
                           {'attributes': ['string', 'type', 'relation', 'store', 'required']})

df_campos = pd.DataFrame([
    {
        'campo': nombre,
        'etiqueta': info.get('string', ''),
        'tipo': info.get('type', ''),
        'relacion': info.get('relation', ''),
        'almacenado': info.get('store', False),
        'requerido': info.get('required', False),
    }
    for nombre, info in campos.items()
]).sort_values('campo').reset_index(drop=True)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
print(f"Total de campos: {len(df_campos)}")
df_campos

Total de campos: 142


,campo,etiqueta,tipo,relacion,almacenado,requerido
0,active,Active,boolean,,True,False
1,activity_calendar_event_id,Next Activity Calendar Event,many2one,calendar.event,False,False
2,activity_date_deadline,Next Activity Deadline,date,,False,False
3,activity_exception_decoration,Activity Exception Decoration,selection,,False,False
4,activity_exception_icon,Icon,char,,False,False
5,activity_ids,Activities,one2many,mail.activity,True,False
6,activity_state,Activity State,selection,,False,False
7,activity_summary,Next Activity Summary,char,,False,False
8,activity_type_icon,Activity Type Icon,char,,False,False
9,activity_type_id,Next Activity Type,many2one,mail.activity.type,False,False


In [7]:
import os
import xmlrpc.client
import pandas as pd
from dotenv import load_dotenv

# Cargar variables de entorno (.env)
load_dotenv()

# ── Conexión a Odoo ────────────────────────────────────────────────────────────
def conectar_odoo():
    """Conecta a la API XML-RPC de Odoo y devuelve credenciales y object proxy."""
    url  = os.getenv("url")
    db   = os.getenv("db")
    user = os.getenv("username_odoo")
    pw   = os.getenv("password")
    
    common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")
    uid = common.authenticate(db, user, pw, {})
    
    if not uid:
        raise RuntimeError("Autenticación Odoo fallida. Verifica variables de entorno.")
    
    models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object")
    return db, uid, pw, models


def main():
    # 1. Nos conectamos a Odoo
    db, uid, pw, models = conectar_odoo()
    
    # 2. Definir el modelo a consultar (en tu ejemplo era 'hr.employee.public')
    modelo = 'hr.employee.public'
    print(f"Obteniendo campos del modelo '{modelo}'...")

    # 3. Consulta de campos (fields_get)
    campos = models.execute_kw(
        db, uid, pw, modelo, 'fields_get', [],
        {'attributes': ['string', 'type', 'relation', 'store', 'required']}
    )

    # 4. Construcción del DataFrame
    df_campos = pd.DataFrame([
        {
            'campo': nombre,
            'etiqueta': info.get('string', ''),
            'tipo': info.get('type', ''),
            'relacion': info.get('relation', ''),
            'almacenado': info.get('store', False),
            'requerido': info.get('required', False),
        }
        for nombre, info in campos.items()
    ]).sort_values('campo').reset_index(drop=True)

    # 5. Ajustar vista en consola y mostrar resultados
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_colwidth', None)
    print(f"Total de campos: {len(df_campos)}")
    print(df_campos.head()) # Muestra solo los primeros para no saturar la consola

    # 6. Exportar el DataFrame final a un archivo (CSV / Excel)
    nombre_archivo = "campos_odoo.csv"
    df_campos.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
    # df_campos.to_excel("campos_odoo.xlsx", index=False)
    
    print(f"\nDataFrame exportado correctamente a: {nombre_archivo}")


if __name__ == "__main__":
    main()

Obteniendo campos del modelo 'hr.employee.public'...
Total de campos: 98
              campo                           etiqueta      tipo     relacion  \
0            active                             Active   boolean                
1        address_id                       Work Address  many2one  res.partner   
2               afc  Valor aporte mensual a cuenta AFC   integer                
3          afp_code                         Codigo AFP      char                
4  allocation_count    Total number of days allocated.     float                

   almacenado  requerido  
0        True      False  
1        True      False  
2       False      False  
3       False      False  
4       False      False  

DataFrame exportado correctamente a: campos_odoo.csv
